# Preview `te_process.csv`

Load and print the first 10 rows. Later cells scan the full CSV for unique-value counts and min/max ranges.

In [9]:
from pathlib import Path

import pandas as pd

print("working")
csv_path = Path("te_process.csv")
df = pd.read_csv(csv_path, nrows=10)
pd.set_option("display.max_columns", None)
print(df)


working
   faultNumber  simulationRun  sample  xmeas_1  xmeas_2  xmeas_3  xmeas_4  \
0          0.0            1.0       1  0.25038   3674.0   4529.0   9.2320   
1          0.0            1.0       2  0.25109   3659.4   4556.6   9.4264   
2          0.0            1.0       3  0.25038   3660.3   4477.8   9.4426   
3          0.0            1.0       4  0.24977   3661.3   4512.1   9.4776   
4          0.0            1.0       5  0.29405   3679.0   4497.0   9.3381   
5          0.0            1.0       6  0.29303   3691.7   4502.2   9.3780   
6          0.0            1.0       7  0.24301   3658.8   4541.6   9.3374   
7          0.0            1.0       8  0.24090   3653.3   4500.0   9.3495   
8          0.0            1.0       9  0.29416   3654.3   4454.7   9.3213   
9          0.0            1.0      10  0.29372   3675.9   4487.4   9.4107   

   xmeas_5  xmeas_6  xmeas_7  xmeas_8  xmeas_9  xmeas_10  xmeas_11  xmeas_12  \
0   26.889   42.402   2704.3   74.863   120.41   0.33818    80.0

In [10]:
print("=== unique count per column (full CSV) ===")
uniques = {}
for chunk in pd.read_csv(csv_path, chunksize=200_000):
    for col in chunk.columns:
        uniques.setdefault(col, set()).update(chunk[col].dropna().unique())

for col, values in uniques.items():
    print(f"{col}: {len(values)}")


=== unique count per column (full CSV) ===
faultNumber: 21
simulationRun: 500
sample: 960
xmeas_1: 305017
xmeas_2: 4807
xmeas_3: 14017
xmeas_4: 29636
xmeas_5: 2820
xmeas_6: 4217
xmeas_7: 5538
xmeas_8: 21262
xmeas_9: 139
xmeas_10: 68547
xmeas_11: 15751
xmeas_12: 8353
xmeas_13: 5895
xmeas_14: 10505
xmeas_15: 8640
xmeas_16: 5635
xmeas_17: 6261
xmeas_18: 20038
xmeas_19: 122563
xmeas_20: 13403
xmeas_21: 18924
xmeas_22: 18426
xmeas_23: 15548
xmeas_24: 24202
xmeas_25: 16932
xmeas_26: 15055
xmeas_27: 12863
xmeas_28: 18207
xmeas_29: 22835
xmeas_30: 3393
xmeas_31: 24985
xmeas_32: 63015
xmeas_33: 17538
xmeas_34: 15231
xmeas_35: 28172
xmeas_36: 15730
xmeas_37: 223331
xmeas_38: 61903
xmeas_39: 62907
xmeas_40: 5453
xmeas_41: 5705
xmv_1: 45344
xmv_2: 67098
xmv_3: 230849
xmv_4: 63452
xmv_5: 97020
xmv_6: 189224
xmv_7: 23169
xmv_8: 18990
xmv_9: 184310
xmv_10: 193879
xmv_11: 133491
source: 2
fault_status: 2


In [11]:
print("=== min / max per column (full CSV) ===")
mins = None
maxs = None
for chunk in pd.read_csv(csv_path, chunksize=200_000):
    chunk_min = chunk.min()
    chunk_max = chunk.max()
    if mins is None:
        mins, maxs = chunk_min, chunk_max
    else:
        mins = mins.combine(chunk_min, min)
        maxs = maxs.combine(chunk_max, max)

for col in mins.index:
    print(f"{col}: min={mins[col]}, max={maxs[col]}")


=== min / max per column (full CSV) ===
faultNumber: min=0.0, max=20.0
simulationRun: min=1.0, max=500.0
sample: min=1, max=960
xmeas_1: min=-0.0049855, max=1.0175
xmeas_2: min=3308.4, max=3906.7
xmeas_3: min=3540.7, max=5175.8
xmeas_4: min=6.6399, max=12.24
xmeas_5: min=25.348, max=28.565
xmeas_6: min=39.656, max=44.653
xmeas_7: min=2413.8, max=3000.5
xmeas_8: min=61.132, max=87.189
xmeas_9: min=119.61, max=121.01
xmeas_10: min=0.018396, max=0.82073
xmeas_11: min=68.097, max=87.591
xmeas_12: min=44.627, max=55.481
xmeas_13: min=2317.1, max=2945.4
xmeas_14: min=18.436, max=33.092
xmeas_15: min=44.311, max=55.912
xmeas_16: min=2870.4, max=3452.7
xmeas_17: min=19.137, max=27.239
xmeas_18: min=52.119, max=74.699
xmeas_19: min=-3.5372, max=466.71
xmeas_20: min=230.15, max=400.71
xmeas_21: min=79.898, max=100.28
xmeas_22: min=62.636, max=83.808
xmeas_23: min=23.225, max=40.211
xmeas_24: min=7.4438, max=10.345
xmeas_25: min=16.909, max=36.469
xmeas_26: min=5.8989, max=7.8854
xmeas_27: min=12

In [12]:
n_lines = 0
with csv_path.open("rb") as f:
    for buf in iter(lambda: f.read(1024 * 1024), b""):
        n_lines += buf.count(b"\n")

n_rows = n_lines - 1  # subtract header
print(f"rows: {n_rows}")


rows: 15330000


# BibMon split on this CSV

[`bibmon.load_tennessee_eastman(train_id=0, test_id=1)`](https://bibmon.readthedocs.io/en/latest/tutorial_tep.html) is a **protocol**, not something you must call. This file is the large Rieth TEP set (500 runs), not Chiang’s 22 files.

- Train on **normal** data: `source=="train"`, `faultNumber==0`
- Test on **IDV(1)**: `source=="test"`, `faultNumber==1`
- Sampling is **3 minutes**. Test faults inject after 8 h → **sample 161** → `2020-02-01 08:00:00`
- Train *fault* files inject after 1 h → **sample 21**. Do not reuse 08:00 there.

`fault_status` is run-level (all 960 test IDV(1) rows say `faulty`). Build labels from `sample`, not from that column.

In [13]:
import numpy as np

FEATURE_COLS = [c for c in pd.read_csv(csv_path, nrows=0).columns if c.startswith("xmeas_") or c.startswith("xmv_")]
META_COLS = ["faultNumber", "simulationRun", "sample", "source", "fault_status"]

TEST_START = pd.Timestamp("2020-02-01 00:00:00")
FAULT_START = pd.Timestamp("2020-02-01 08:00:00")  # 8 h, sample 161
TRAIN_FAULT_SAMPLE = 21  # Rieth train files: 1 h
TEST_FAULT_SAMPLE = 161


def load_runs(faults=(0, 1), run=1):
    """Stream the 5.6 GB CSV and keep a few complete trajectories."""
    chunks = []
    usecols = META_COLS + FEATURE_COLS
    for chunk in pd.read_csv(csv_path, usecols=usecols, chunksize=400_000):
        keep = chunk["simulationRun"].eq(run) & chunk["faultNumber"].isin(faults)
        if keep.any():
            chunks.append(chunk.loc[keep])
    out = pd.concat(chunks, ignore_index=True)
    out["faultNumber"] = out["faultNumber"].astype(int)
    out["simulationRun"] = out["simulationRun"].astype(int)
    return out.sort_values(["source", "faultNumber", "sample"]).reset_index(drop=True)


def add_clock(df, origin):
    """BibMon-style synthetic DatetimeIndex: 3 min / sample, sample 1 at origin."""
    out = df.copy()
    out["time"] = origin + pd.to_timedelta((out["sample"] - 1) * 3, unit="min")
    return out.set_index("time")


tep = load_runs()
print(tep.groupby(["source", "faultNumber", "fault_status"]).agg(n=("sample", "size"), smin=("sample", "min"), smax=("sample", "max")))

                                   n  smin  smax
source faultNumber fault_status                 
test   0           normal        960     1   960
       1           faulty        960     1   960
train  0           normal        500     1   500
       1           faulty        500     1   500


In [14]:
# Toy 1 — same objects as the BibMon tutorial, from this CSV
df_train = add_clock(
    tep[(tep.source == "train") & (tep.faultNumber == 0)],
    pd.Timestamp("2020-01-01"),
)
df_test = add_clock(
    tep[(tep.source == "test") & (tep.faultNumber == 1)],
    TEST_START,
)
fault_start = "2020-02-01 08:00:00"

print("train", df_train.shape, df_train.index.min(), "→", df_train.index.max())
print("test ", df_test.shape, df_test.index.min(), "→", df_test.index.max())
print("iloc of fault_start:", df_test.index.get_loc(fault_start), "(BibMon axvline(160))")
print("sample at fault_start:", int(df_test.loc[fault_start, "sample"]))

# Datetime slicing is the analysis convenience
pre, post = df_test.loc[:fault_start], df_test.loc[fault_start:]
print("\nIDV(1) mean xmeas_1 / xmv_3")
print("  pre 08:00 ", pre["xmeas_1"].mean().round(4), pre["xmv_3"].mean().round(2))
print("  post 08:00", post["xmeas_1"].mean().round(4), post["xmv_3"].mean().round(2))

train (500, 57) 2020-01-01 00:00:00 → 2020-01-02 00:57:00
test  (960, 57) 2020-02-01 00:00:00 → 2020-02-02 23:57:00
iloc of fault_start: 160 (BibMon axvline(160))
sample at fault_start: 161

IDV(1) mean xmeas_1 / xmv_3
  pre 08:00  0.2495 24.54
  post 08:00 0.7597 74.74


In [15]:
# Toy 2 — fault_status is the wrong label; rebuild y from the clock
test = tep[tep.source == "test"].copy()
test["y_csv"] = (test["fault_status"] == "faulty").astype(int)
test["y_true"] = ((test["faultNumber"] > 0) & (test["sample"] >= TEST_FAULT_SAMPLE)).astype(int)

print("CSV label vs true label (test, run 1, faults 0 and 1)")
print(pd.crosstab(test["y_csv"], test["y_true"], rownames=["fault_status"], colnames=["true"]))

poisoned = (test["y_csv"] == 1) & (test["y_true"] == 0)
print(f"\npoisoned 'faulty' rows that are still healthy: {poisoned.sum()} (should be 160 for IDV(1))")
print(test.loc[poisoned, ["faultNumber", "sample", "fault_status"]].head(3))
print("...")
print(test.loc[poisoned, ["faultNumber", "sample", "fault_status"]].tail(3))

CSV label vs true label (test, run 1, faults 0 and 1)
true            0    1
fault_status          
0             960    0
1             160  800

poisoned 'faulty' rows that are still healthy: 160 (should be 160 for IDV(1))
     faultNumber  sample fault_status
960            1       1       faulty
961            1       2       faulty
962            1       3       faulty
...
      faultNumber  sample fault_status
1117            1     158       faulty
1118            1     159       faulty
1119            1     160       faulty


In [16]:
# Toy 3 — unsupervised PCA-SPE detector (BibMon's SPE idea, numpy only)
X_train = df_train[FEATURE_COLS].to_numpy()
mu, sd = X_train.mean(0), X_train.std(0)
sd[sd == 0] = 1.0
Z_train = (X_train - mu) / sd

_, S, Vt = np.linalg.svd(Z_train, full_matrices=False)
explained = np.cumsum(S**2) / (S**2).sum()
n_comp = int(np.searchsorted(explained, 0.90) + 1)
P = Vt[:n_comp].T  # 52 x n_comp

spe_train = ((Z_train - Z_train @ P @ P.T) ** 2).sum(1)
limit = np.quantile(spe_train, 0.99)
print(f"PCs for 90% variance: {n_comp}  SPE 99% limit: {limit:.2f}")


def score(frame):
    Z = (frame[FEATURE_COLS].to_numpy() - mu) / sd
    spe = ((Z - Z @ P @ P.T) ** 2).sum(1)
    alarm = spe > limit
    return pd.Series(spe, index=frame.index, name="SPE"), pd.Series(alarm, index=frame.index, name="alarm")


spe, alarm = score(df_test)
pre_mask = df_test.index < fault_start
far = alarm.loc[pre_mask].mean()
fdr = alarm.loc[fault_start:].mean()
hits = alarm.loc[fault_start:]
first_hit = hits.index[hits.to_numpy()][0] if hits.any() else None
print(f"FAR before {fault_start}: {far:.3f}")
print(f"FDR after  {fault_start}: {fdr:.3f}")
print("first alarm at/after fault_start:", first_hit)
print(spe.loc[fault_start:].head(8).round(2))

PCs for 90% variance: 30  SPE 99% limit: 12.30
FAR before 2020-02-01 08:00:00: 0.231
FDR after  2020-02-01 08:00:00: 0.999
first alarm at/after fault_start: 2020-02-01 08:03:00
time
2020-02-01 08:00:00     4.33
2020-02-01 08:03:00    15.81
2020-02-01 08:06:00    22.13
2020-02-01 08:09:00    38.95
2020-02-01 08:12:00    49.15
2020-02-01 08:15:00    57.19
2020-02-01 08:18:00    71.61
2020-02-01 08:21:00    97.88
Name: SPE, dtype: float64


In [17]:
# Toy 4 — time-series EDA: rolling mean + hourly resample around the changepoint
window = df_test.loc["2020-02-01 06:00":"2020-02-01 12:00", ["xmeas_1", "xmv_3"]]
window = window.assign(xmeas_1_roll=window["xmeas_1"].rolling("30min").mean())

hourly = df_test[["xmeas_1", "xmv_3"]].resample("1h").mean()
print("hourly A-feed around the step:")
print(hourly.loc["2020-02-01 05:00":"2020-02-01 14:00", "xmeas_1"].round(3))

# Same run, healthy test file: first 8 h are bit-identical to IDV(1)
df_healthy = add_clock(
    tep[(tep.source == "test") & (tep.faultNumber == 0)],
    TEST_START,
)
pre_cols = FEATURE_COLS
identical = np.allclose(
    df_test.loc[:fault_start].iloc[:-1][pre_cols],
    df_healthy.loc[:fault_start].iloc[:-1][pre_cols],
)
print("\npre-fault test IDV(0) == IDV(1) for this simulationRun?", identical)

hourly A-feed around the step:
time
2020-02-01 05:00:00    0.241
2020-02-01 06:00:00    0.220
2020-02-01 07:00:00    0.272
2020-02-01 08:00:00    0.397
2020-02-01 09:00:00    0.718
2020-02-01 10:00:00    0.927
2020-02-01 11:00:00    0.952
2020-02-01 12:00:00    0.819
2020-02-01 13:00:00    0.648
2020-02-01 14:00:00    0.727
Freq: h, Name: xmeas_1, dtype: float64

pre-fault test IDV(0) == IDV(1) for this simulationRun? True


In [18]:
# Toy 5 — supervised threshold on xmeas_1: true labels vs poisoned CSV labels
x = df_test["xmeas_1"]
y_true = (df_test.index >= fault_start).astype(int)
y_poison = np.ones(len(df_test), dtype=int)  # what fault_status claims for IDV(1)

# threshold from normal train (mean + 4 sd)
thr = df_train["xmeas_1"].mean() + 4 * df_train["xmeas_1"].std()
y_hat = (x > thr).astype(int)


def prf(y, yhat, mask=None):
    if mask is not None:
        y, yhat = y[mask], yhat[mask]
    tp = int(((y == 1) & (yhat == 1)).sum())
    fp = int(((y == 0) & (yhat == 1)).sum())
    fn = int(((y == 1) & (yhat == 0)).sum())
    prec = tp / (tp + fp) if tp + fp else 0.0
    rec = tp / (tp + fn) if tp + fn else 0.0
    return prec, rec


print(f"xmeas_1 threshold from train normal: {thr:.3f}")
print("metrics on TRUE labels (healthy prefix + faulty suffix):  P={:.3f} R={:.3f}".format(*prf(y_true, y_hat)))
print("metrics if you believed fault_status (all 960 = faulty):   P={:.3f} R={:.3f}".format(*prf(y_poison, y_hat)))
print("that second row looks 'fine' only because it never checks the first 8 hours.")

xmeas_1 threshold from train normal: 0.363
metrics on TRUE labels (healthy prefix + faulty suffix):  P=1.000 R=0.988
metrics if you believed fault_status (all 960 = faulty):   P=1.000 R=0.823
that second row looks 'fine' only because it never checks the first 8 hours.
